# 05 - Data Validation

Skipping ahead to Notebook 5 in the sprint. This one is about writing actual rules to catch bad data - not just eyeballing `.describe()` output, but checking every row against conditions that should always hold true, and getting a count of exactly how many rows break each rule.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_churn.csv")
df.shape

(5060, 14)

## 1. What is Data Validation

Data validation means checking whether the data follows a set of rules I define upfront, based on what's actually possible or sensible for that field. It's different from just looking for missing values - a row can be 100% filled in and still be invalid (age = 200 is not missing, it's wrong).

I'm treating validation as a *checking* step here, not a *fixing* step - the point is to measure how big each problem is and flag exactly which rows fail, before deciding what to do about them in the cleaning notebooks.

**Business example:** an e-commerce order form might accept "quantity = -3" if there's no validation rule stopping it, and that negative quantity would then mess up every downstream inventory and revenue calculation. A validation rule ("quantity must be >= 1") catches that before it ever reaches a report.

**AI/ML relevance:** if I train a model on unvalidated data, it will treat impossible values as real patterns. A model that sees a handful of "age = 200" rows might start assigning weird importance to extreme ages, when really those rows are just noise from bad input.

## 2. Range Validation

Checks that a numeric value falls inside a sensible min/max. For this dataset: `Age` should realistically be between 18 and 90 (this is a customer base, not newborns or 150-year-olds), and `MonthlyCharges`/`TotalCharges` should never be negative.

In [2]:
range_rules = {
    "Age": (18, 90),
    "MonthlyCharges": (0, None),
    "TotalCharges_numeric": (0, None),
}

age_violations = df[(df["Age"] < 18) | (df["Age"] > 90)]
print("Age out of range:", len(age_violations))
print(age_violations["Age"].unique())

charges_violations = df[df["MonthlyCharges"] < 0]
print("\nNegative MonthlyCharges rows:", len(charges_violations))

Age out of range: 10
[120. 150.  -5.]

Negative MonthlyCharges rows: 0


## 3. Type Validation

Checks that a column is actually stored as the data type it should be. `TotalCharges` should be numeric, but earlier I saw it's stored as `object` because of blank string entries. Type validation here is really about confirming *why* it's object-typed - trying to force it numeric and seeing which values fail the conversion.

In [3]:
# attempt numeric conversion, anything that can't convert becomes NaN,
# and those NaNs (that weren't already NaN) tell me the actual bad values
converted = pd.to_numeric(df["TotalCharges"], errors="coerce")
newly_invalid = df[converted.isna() & df["TotalCharges"].notna()]

print("Rows where TotalCharges is not truly numeric:", len(newly_invalid))
newly_invalid["TotalCharges"].unique()

Rows where TotalCharges is not truly numeric: 0


array([], dtype=float64)

## 4. Format Validation

Checks that a value follows an expected pattern - not just any string, a specific shape. `CustomerID` in this dataset should always look like `CUST` followed by 4 digits (e.g. `CUST1000`). If any row breaks that pattern, it's likely a data entry or import error.

In [4]:
import re

pattern = re.compile(r"^CUST\d{4}$")
bad_ids = df[~df["CustomerID"].astype(str).str.match(pattern)]
print("CustomerID rows not matching expected format:", len(bad_ids))

CustomerID rows not matching expected format: 5060


## 5. Category Validation

Checks that a categorical column only contains values from an accepted, known set. This is where the messy spellings I flagged back in Notebook 1 (`Gender`: "Male"/"M"/"male", `ContractType`: "Month-to-month"/"month-to-month"/"Monthly") show up as validation failures, even though technically none of these values are "wrong" in isolation - they're just not part of one consistent accepted set.

In [5]:
accepted_gender = {"Male", "Female"}
accepted_contract = {"Month-to-month", "One year", "Two year"}

bad_gender = df[~df["Gender"].isin(accepted_gender)]
bad_contract = df[~df["ContractType"].isin(accepted_contract)]

print("Gender values outside accepted set:")
print(bad_gender["Gender"].value_counts())

print("\nContractType values outside accepted set:")
print(bad_contract["ContractType"].value_counts())

Gender values outside accepted set:
Gender
male    1283
M       1256
Name: count, dtype: int64

ContractType values outside accepted set:
ContractType
Two Year          1710
Month-to-Month    1707
One Year          1643
Name: count, dtype: int64


## 6. Null Validation

Checks which columns are actually allowed to have missing values and which ones absolutely shouldn't. For example, `Churn` (the target column) should never be null - a row with no label is useless for training and needs to be dropped or investigated, not imputed.

In [6]:
required_columns = ["CustomerID", "Churn", "ContractType"]

for col in required_columns:
    null_count = df[col].isnull().sum()
    print(f"{col}: {null_count} nulls (should be 0)")

CustomerID: 0 nulls (should be 0)
Churn: 0 nulls (should be 0)
ContractType: 0 nulls (should be 0)


## 7. Business Rule Validation

These are rules that come from domain knowledge, not just column-by-column checks. For this dataset, a sensible business rule is: `TotalCharges` should be roughly consistent with `MonthlyCharges * TenureYears * 12`. If a row is wildly off from that expected relationship, something's wrong with one of the three fields, even if each field passes its own individual range check.

In [7]:
df_check = df.copy()
df_check["TotalCharges_numeric"] = pd.to_numeric(df_check["TotalCharges"], errors="coerce")

expected_total = df_check["MonthlyCharges"] * df_check["TenureYears"] * 12
diff_ratio = (df_check["TotalCharges_numeric"] - expected_total).abs() / expected_total.replace(0, np.nan)

business_rule_violations = df_check[diff_ratio > 0.5]  # more than 50% off expected
print("Rows where TotalCharges deviates heavily from MonthlyCharges * TenureYears * 12:", len(business_rule_violations))

Rows where TotalCharges deviates heavily from MonthlyCharges * TenureYears * 12: 2981


## 8. Referential Validation

Checks that values referencing another table/column/known list are actually valid. This dataset is a single flat table so there's no foreign-key relationship to check, but the equivalent idea here is `InternetService` and `TechSupport` - a row saying `InternetService = "No"` but `TechSupport = "Yes"` doesn't make sense, since you can't have tech support for internet you don't have. That's a referential/logical consistency issue between two columns.

In [8]:
referential_violation = df[(df["InternetService"] == "No") & (df["TechSupport"] == "Yes")]
print("Rows with TechSupport = Yes but no InternetService:", len(referential_violation))

Rows with TechSupport = Yes but no InternetService: 880


## 9. Date Validation

This dataset doesn't actually have date columns (tenure is stored as years, not join/leave dates), so I can't run this against real data here. But the rule itself matters for the sprint brief's example: **date of joining cannot be after date of leaving, and date of birth cannot be after date of joining.** If this dataset had `JoinDate` and `DOB` columns, the check would look like this:

\`\`\`python
invalid_dates = df[df["JoinDate"] < df["DOB"]]
\`\`\`

Flagging this because it's a real validation category, just not one this particular dataset needs - not every dataset needs every rule.

## 10. Constraint Validation

A catch-all for rules that don't neatly fit the categories above - things like uniqueness constraints. `CustomerID` should be unique per customer; if the same ID shows up more than once, that's either a duplicate record or two different customers wrongly sharing an ID.

In [9]:
dup_ids = df[df["CustomerID"].duplicated(keep=False)]
print("Rows sharing a CustomerID with another row:", len(dup_ids))
print("Unique CustomerIDs involved:", dup_ids["CustomerID"].nunique())

Rows sharing a CustomerID with another row: 80
Unique CustomerIDs involved: 40


## Pulling it together - a validation summary

Instead of scattering these checks across the sprint, it's useful to run all of them in one place and get a single report. This is basically what I'd hand off if someone asked "how bad is this dataset" before cleaning starts.

validation_report = {
    "Age out of range (18-90)": len(age_violations),
    "TotalCharges not truly numeric": len(newly_invalid),
    "CustomerID bad format": len(bad_ids),
    "Gender outside accepted set": len(bad_gender),
    "ContractType outside accepted set": len(bad_contract),
    "Churn nulls": df["Churn"].isnull().sum(),
    "TotalCharges vs expected mismatch": len(business_rule_violations),
    "TechSupport without InternetService": len(referential_violation),
    "Duplicate CustomerID rows": len(dup_ids),
}

report_df = pd.DataFrame(list(validation_report.items()), columns=["Rule", "Rows Failing"])
report_df